# StoCFL: efeito do dropout com retorno no aprendizado federado, com rótulos non-IID

Roda o StoCFL completo (`StoCFL-FCC+CFL.py --train 1`) em MNIST com **rótulos non-IID por Dirichlet** (`--dirichlet_alpha`, sem rotação, `k = 1`), **com e sem dropout com retorno**, e mede a clusterização e a **acurácia** dos modelos de cluster e do modelo global. É o passo 4 do [plano do dropout com treino](../../docs/plano-dropout-treino-stocfl.md) (D-043 a D-047 em [decisoes.md](../../docs/decisoes.md)).

- **Grade:** `α` em `ALPHAS` (0.5 e 1.0) × `m` em `FRACOES` (100% e 50% dos 240 clientes) × `d` em `DROPS` (sem dropout, 0.05 e 0.1, ausências de 1 a 10 rodadas) × 10 seeds = **120 execuções** de 50 rodadas. Treino do artigo (Rotated MNIST): `lr = 0.1`, 5 épocas, `μ = 0.05`, batch cheio. `τ = 0.84683`, escolhido por Monte Carlo no [varredura-tau-dirichlet.ipynb](varredura-tau-dirichlet.ipynb) (D-046).
- **O algoritmo não recebe o nº de clusters**, e não há grupos reais: o nº e o tamanho dos clusters são resultados. Não há ARI.
- **Avaliação:** com `k = 1`, todo modelo é testado no MNIST de teste inteiro (10 classes balanceadas). Um modelo de cluster especializado em poucas classes mede **pior** nesse teste, por construção (plano, seção 7). Por isso o notebook calcula também a **acurácia ponderada**: `Σ_c p_c · recall_c`, com `p` a distribuição de rótulos do cluster (soma dos histogramas dos membros) e o recall por classe gravado pelo script no `accuracy.pkl` (D-045).
- **Política 7:** as execuções gravam logs em `logs/`; a última seção grava `logs/resultados-dropout-treino.json` com os parâmetros, as métricas e os dados brutos de cada execução (partição, sorteados e indisponíveis por rodada, acurácia e recall de cada cluster avaliado, cossenos da última rodada e o histograma de rótulos dos clientes). Com `DADOS_JSON = True` o notebook só lê o JSON e plota. Sem as execuções e com o JSON destes parâmetros já gravado, ele para com um aviso em vez de rodar tudo de novo (`REEXECUTAR = True` força).
- **Ambiente:** o kernel é o `.venv` da raiz; abra o notebook a partir de `projetos/StoCFL/`. As partições ficam em `datasets/rotated_mnist_<N>_<seed>_k1_a<α>/`.

**Custo:** cada execução grava `psi.pkl` (~1,5 GB com 240 clientes) a cada rodada; `APAGAR_PSI = True` (padrão) o apaga ao fim de cada execução, depois de conferir a cobertura. O tempo por execução foi medido no smoke test (guia, seção 4.11).

| Métrica | Definição |
|---|---|
| `acc_cluster`, `acc_global` | por rodada, média, ponderada pelos participantes de cada cluster avaliado, da acurácia do modelo do cluster e do `gmodel` no teste |
| `acc_cluster_pond`, `acc_global_pond` | o mesmo com a acurácia ponderada pela distribuição de rótulos do cluster |
| `*_final` | média das últimas `JANELA_FINAL` rodadas |
| `ganho`, `ganho_pond` | `acc_cluster − acc_global` (e a versão ponderada), no valor final |
| `rodada_limiar` | primeira rodada (1-based) com `acc_cluster ≥ LIMIAR_ACC`; vazia se não chegar |
| `clusters`, `maior`, `singletons`, `tvd` | da partição: nº de clusters, maior cluster, clusters de 1 cliente e a TVD média entre a distribuição de rótulos do cliente e a do seu cluster |
| `estavel` | primeira rodada a partir da qual a partição não muda mais |

In [ ]:
import json
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from scipy.stats import wilcoxon

PROJETO = Path.cwd()
SCRIPT = "StoCFL-FCC+CFL.py"
assert (PROJETO / SCRIPT).exists(), "abra o notebook a partir de projetos/StoCFL/"
RESULTADOS_JSON = PROJETO / "logs" / "resultados-dropout-treino.json"

# Parâmetros da grade (plano, seção 2; D-043, D-044, D-046)
N = 240                           # nº de clientes
ALPHAS = [0.5, 1.0]               # α do Dirichlet (--dirichlet_alpha)
ROUNDS = 50                       # rodadas de comunicação
TAU = 0.84683                     # limiar de cosseno (--tau), escolhido por Monte Carlo (varredura-tau-dirichlet.ipynb, D-046)
MU, LR, EPOCHS = 0.05, 0.1, 5     # treino do artigo no Rotated MNIST (Tabela 1); batch cheio (sem --batch_size)
SEEDS = list(range(10))           # a seed define a partição dos dados, o modelo inicial e o sorteio
FRACOES = [1.0, 0.5]              # m: fração dos clientes sorteada por rodada
DROPS = [0.0, 0.05, 0.1]          # d (--drop); 0.0 = StoCFL sem dropout
DROP_ROUNDS = 10                  # L_max (--drop_rounds)
JANELA_FINAL = 10                 # rodadas no fim que dão o valor final das acurácias (41 a 50)
LIMIAR_ACC = 0.90                 # limiar da rodada_limiar (ajustar depois de ver as curvas)
REEXECUTAR = False                # True ignora execuções já concluídas e roda de novo
APAGAR_PSI = True                 # True apaga o psi.pkl (~1,5 GB) de cada execução concluída, depois de conferir a cobertura
DADOS_JSON = False                # True não lê logs nem roda nada: carrega tudo de RESULTADOS_JSON e só plota

assert 0.0 in DROPS and all(0 <= d < 1 for d in DROPS) and all(int(N * f) >= 1 for f in FRACOES)


def nome(d):
    return "sem dropout" if d == 0 else f"d = {d:g}"


def parametros_atuais():
    return {"N": N, "ALPHAS": ALPHAS, "ROUNDS": ROUNDS, "TAU": TAU, "MU": MU, "LR": LR, "EPOCHS": EPOCHS, "SEEDS": SEEDS,
            "FRACOES": FRACOES, "DROPS": DROPS, "DROP_ROUNDS": DROP_ROUNDS, "JANELA_FINAL": JANELA_FINAL, "LIMIAR_ACC": LIMIAR_ACC}


print("GPU:", torch.cuda.is_available(), "| clientes por rodada:", {f: int(N * f) for f in FRACOES},
      "| disponibilidade média:", {nome(d): round(1 / (1 + d * (DROP_ROUNDS + 1) / 2), 3) for d in DROPS})

## 1. Execução

`executar(α, frac, seed, drop)` roda o script uma vez com `--train 1 --k 1 --dirichlet_alpha α`. Uma execução concluída é reaproveitada se o `cluster.pkl` e o `accuracy.pkl` tiverem `ROUNDS` entradas **e** a linha de argumentos do log tiver os mesmos parâmetros (o nome da pasta não traz o `τ` nem o treino). A saída padrão vai para `logs/pipeline-dropout-treino-*.txt`, e é conferida (seção 2) antes de os logs serem apagados. Antes de apagar o `psi.pkl`, a célula confere que as chaves dele são os clientes já sorteados (cobertura, I5 do plano do dropout).

O nome da pasta de logs tem a hora só até o minuto: `executar` espera o minuto mudar se já houver uma pasta da mesma combinação naquele minuto (D-045).

In [ ]:
def dados_prontos(alpha, seed):
    d = PROJETO / f"datasets/rotated_mnist_{N}_{seed}_k1_a{alpha}"
    return len(list((d / "train").glob("data*.pkl"))) == N and (d / "test" / "data0.pkl").exists()


def pastas_de_log(alpha, m, seed, drop):
    tag = f"-drop{drop}-r{DROP_ROUNDS}" if drop > 0 else ""
    padrao = f"runs-rotated-mnist-n{N}-p{m}-seed{seed}-lambda{MU}-k1-a{alpha}{tag}-time-*"
    return sorted((PROJETO / "logs").glob(padrao), key=lambda p: p.stat().st_mtime)


def args_do_log(pasta):
    for linha in (pasta / "rotated_mnist.log").read_text().splitlines():
        if "Namespace(" in linha:
            return dict(re.findall(r"(\w+)=([^,()]+)", linha.split("Namespace(", 1)[1]))
    return {}


def saida_de(alpha, m, seed, drop):
    return PROJETO / "logs" / f"pipeline-dropout-treino-a{alpha}-m{m}-seed{seed}-drop{drop}.txt"


def concluida(pasta, alpha, m, seed, drop):
    try:
        if len(torch.load(pasta / "cluster.pkl", weights_only=False)) != ROUNDS:
            return False
        if len(torch.load(pasta / "accuracy.pkl", weights_only=False)) != ROUNDS:
            return False
        a = args_do_log(pasta)
        esperado = {"n": N, "com_round": ROUNDS, "num_per_round": m, "mu": MU, "lr": LR, "epochs": EPOCHS, "tau": TAU, "train": 1,
                    "seed": seed, "k": 1, "dirichlet_alpha": alpha, "drop": drop}
        if a.get("batch_size") != "None" or any(float(a[c]) != v for c, v in esperado.items()):
            return False
        return drop == 0 or int(a["drop_rounds"]) == DROP_ROUNDS
    except Exception:
        return False


def pronta(alpha, frac, seed, drop):
    m = int(N * frac)
    return next((p for p in reversed(pastas_de_log(alpha, m, seed, drop)) if concluida(p, alpha, m, seed, drop)), None)


def executar(alpha, frac, seed, drop):
    m = int(N * frac)
    if not REEXECUTAR and (pasta := pronta(alpha, frac, seed, drop)) is not None:
        return pasta, None
    while any(p.name.endswith(time.strftime("-time-%m-%d-%H:%M")) for p in pastas_de_log(alpha, m, seed, drop)):
        time.sleep(5)
    cmd = [sys.executable, SCRIPT, "--n", str(N), "--com_round", str(ROUNDS), "--num_per_round", str(m), "--lr", str(LR),
           "--epochs", str(EPOCHS), "--mu", str(MU), "--tau", str(TAU), "--train", "1", "--seed", str(seed), "--dataset", "mnist",
           "--k", "1", "--dirichlet_alpha", str(alpha), "--process", "0" if dados_prontos(alpha, seed) else "1"]
    if drop > 0:
        cmd += ["--drop", str(drop), "--drop_rounds", str(DROP_ROUNDS)]
    (PROJETO / "logs").mkdir(exist_ok=True)
    saida = saida_de(alpha, m, seed, drop)
    inicio = time.time()
    with open(saida, "w") as f:
        r = subprocess.run(cmd, cwd=PROJETO, stdout=f, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        print(saida.read_text()[-3000:])
        raise RuntimeError(f"falhou: {' '.join(cmd)}")
    pasta = next(p for p in reversed(pastas_de_log(alpha, m, seed, drop)) if p.stat().st_mtime >= inicio - 1)
    assert concluida(pasta, alpha, m, seed, drop), f"{pasta.name}: execução incompleta"
    return pasta, time.time() - inicio


def sorteados_do_log(pasta):
    return [[int(x) for x in a.group(1).split(",") if x.strip()]
            for a in re.finditer(r"client id this round \[(.*?)\]", (pasta / "rotated_mnist.log").read_text())]


def json_com_estes_parametros():
    try:
        return json.loads(RESULTADOS_JSON.read_text())["parametros"] == json.loads(json.dumps(parametros_atuais()))
    except (OSError, KeyError, ValueError):
        return False


plano = [(a, f, s, d) for s in SEEDS for a in ALPHAS for f in FRACOES for d in DROPS]
DURACOES, COBERTURA_PSI = {}, {}
if DADOS_JSON:
    DADOS = json.loads(RESULTADOS_JSON.read_text())
    assert DADOS["parametros"] == json.loads(json.dumps(parametros_atuais())), f"a célula 1 difere do JSON, gerado com {DADOS['parametros']}"
    execucoes = {(e["alpha"], e["m"], e["seed"], e["drop"]): e["pasta"] for e in DADOS["execucoes"]}
    assert set(execucoes) == set(plano), "o JSON não tem todas as execuções do plano"
    print(f"{len(execucoes)} execuções lidas de {RESULTADOS_JSON.name} (gerado em {DADOS['gerado_em']}): nada será executado")
else:
    pendentes = plano if REEXECUTAR else [c for c in plano if pronta(*c) is None]
    if pendentes and not REEXECUTAR and json_com_estes_parametros():
        raise RuntimeError(f"{len(pendentes)} execuções sem logs, mas {RESULTADOS_JSON.name} já tem os resultados desta grade: "
                           "use DADOS_JSON = True para plotar sem rodar nada, ou REEXECUTAR = True para rodar a grade de novo")
    psi_gb = N * 6.5e-3
    dados_gb = 0.23 * len({(a, s) for a, _, s, _ in pendentes if not dados_prontos(a, s)}) * N / 240
    necessario = (psi_gb if APAGAR_PSI else psi_gb * len(pendentes)) + dados_gb
    livre = shutil.disk_usage(PROJETO).free / 1e9
    print(f"{len(plano)} execuções no plano, {len(pendentes)} pendentes; espaço estimado {necessario:.0f} GB, livre {livre:.0f} GB")
    assert livre > 1.1 * necessario, "pouco espaço em disco: reduza a grade ou use APAGAR_PSI = True"

    execucoes = {}
    for i, (alpha, frac, seed, drop) in enumerate(plano, start=1):
        pasta, duracao = executar(alpha, frac, seed, drop)
        psi = pasta / "psi.pkl"
        if psi.exists():
            # cobertura do log = chaves de psi.pkl (I5): só dá para conferir antes de apagar
            vistos = set().union(*map(set, sorteados_do_log(pasta)))
            COBERTURA_PSI[str(pasta)] = sorted(torch.load(psi, weights_only=False, map_location="cpu").keys()) == sorted(vistos)
            if APAGAR_PSI:
                psi.unlink()
        execucoes[(alpha, frac, seed, drop)] = pasta
        DURACOES[str(pasta)] = duracao
        estado = "reaproveitada" if duracao is None else f"{duracao:.0f} s"
        print(f"[{i}/{len(plano)}] α={alpha} m={frac:.0%} {nome(drop)} seed={seed}: {estado} -> {pasta.name}")
    assert all(COBERTURA_PSI.values()), "cobertura do log diferente das chaves de psi.pkl"

## 2. Leitura das execuções e métricas por rodada

`ler_execucao` lê, uma vez por pasta: a partição de cada rodada (`cluster.pkl`), os sorteados (log), os indisponíveis (`availability.pkl`), a acurácia e o recall de cada cluster avaliado (`accuracy.pkl`), os ids desses clusters e as acurácias com 4 casas (linhas `Round r, Cluster c` do log) e os cossenos entre clusters da última rodada (`cos_rec.log`). Conferências em cada execução:

- **V3:** a acurácia do log é a do `accuracy.pkl` (a do cluster com 4 casas; a lista do global, exata), com o mesmo nº de clusters por rodada, e os ids do log são os clusters dos sorteados na partição da rodada.
- **V4:** uma entrada por rodada no `cluster.pkl` e no `accuracy.pkl`, inclusive em rodada sem participantes.
- **Saída padrão:** as linhas `Round r, Global acc` da saída padrão são as mesmas do log.

O histograma de rótulos de cada cliente vem dos dados (`datasets/`).

In [ ]:
def ler_execucao(pasta):
    pasta = Path(pasta)
    log = torch.load(pasta / "cluster.pkl", weights_only=False)
    acc = torch.load(pasta / "accuracy.pkl", weights_only=False)
    texto = (pasta / "rotated_mnist.log").read_text()
    sorteados = sorteados_do_log(pasta)
    arq = pasta / "availability.pkl"
    indisp = [[int(c) for c in u] for u in torch.load(arq, weights_only=False)] if arq.exists() else [[] for _ in log]
    assert len(log) == len(acc) == len(sorteados) == len(indisp) == ROUNDS, f"{pasta.name}: nº de rodadas diferente (V4)"
    linhas_cluster = re.findall(r"Round (\d+), Cluster (\d+) - Global Test loss: [\d.]+, acc: ([\d.]+)", texto)
    linhas_global = {int(r): [float(x) for x in g.split(",") if x.strip()] for r, g in re.findall(r"Round (\d+), Global acc: \[(.*?)\]", texto)}
    saida = Path(saida_de(*CHAVE_DE[str(pasta)])) if str(pasta) in CHAVE_DE else None
    if saida is not None and saida.exists():
        assert re.findall(r"Round \d+, Global acc: \[.*?\]", saida.read_text()) == re.findall(r"Round \d+, Global acc: \[.*?\]", texto), \
            f"{pasta.name}: saída padrão diferente do log"
    rodadas_acc, rotulos = [], []
    for r in range(ROUNDS):
        dono = np.empty(N, dtype=int)
        for cid, mem in log[r].items():
            dono[mem] = cid
        rotulos.append(dono.tolist())
        ids = [int(c) for rr, c, _ in linhas_cluster if int(rr) == r]
        e = acc[r]
        assert len(ids) == len(e["cluster"]) == len(e["global"]) == len(e["cluster_recall"]) == len(e["global_recall"]), f"{pasta.name} r{r}: V3"
        assert ids == sorted({int(dono[c]) for c in sorteados[r]}), f"{pasta.name} r{r}: clusters do log ≠ clusters dos sorteados"
        assert all(abs(round(a, 4) - float(x)) < 1e-9 for a, (_, _, x) in zip(e["cluster"], [l for l in linhas_cluster if int(l[0]) == r])), f"{pasta.name} r{r}: V3 cluster"
        assert e["global"] == linhas_global.get(r, []), f"{pasta.name} r{r}: V3 global"   # o log grava a lista com repr completo
        part = np.bincount([dono[c] for c in sorteados[r]], minlength=N)
        rodadas_acc.append([{"cluster": c, "participantes": int(part[c]), "acc_cluster": a, "acc_global": g, "recall_cluster": rc, "recall_global": rg}
                            for c, a, g, rc, rg in zip(ids, e["cluster"], e["global"], e["cluster_recall"], e["global_recall"])])
    arq = pasta / "cos_rec.log"
    achados = [re.search(r"INFO - (\d+),(\d+):(-?[\d.]+)$", l) for l in arq.read_text().splitlines()] if arq.exists() else []
    cos = [[int(a), int(b), float(c)] for a, b, c in (m.groups() for m in achados if m)]
    return {"rotulos_por_rodada": rotulos, "sorteados_por_rodada": sorteados, "indisponiveis_por_rodada": indisp,
            "acuracia_por_rodada": rodadas_acc, "cossenos_entre_clusters": cos}


def histograma(alpha, seed):
    d = PROJETO / f"datasets/rotated_mnist_{N}_{seed}_k1_a{alpha}/train"
    return np.array([np.bincount(np.asarray(torch.load(d / f"data{c}.pkl", weights_only=False).y), minlength=10) for c in range(N)])


CHAVE_DE = {str(p): (a, int(N * f), s, d) for (a, f, s, d), p in execucoes.items()}
if DADOS_JSON:
    BRUTOS = {e["pasta"]: {k: e[k] for k in ("rotulos_por_rodada", "sorteados_por_rodada", "indisponiveis_por_rodada", "acuracia_por_rodada",
                                             "cossenos_entre_clusters")} for e in DADOS["execucoes"]}
    HIST = {(h["alpha"], h["seed"]): np.array(h["histograma"]) for h in DADOS["histogramas"]}
else:
    BRUTOS = {str(p): ler_execucao(p) for p in execucoes.values()}
    HIST = {(a, s): histograma(a, s) for a in ALPHAS for s in SEEDS}
print(f"{len(BRUTOS)} execuções lidas; V3, V4 e a saída padrão conferidas" if not DADOS_JSON else f"{len(BRUTOS)} execuções do JSON")


def rodada_estavel(rotulos):
    # primeira rodada (1-based) a partir da qual a partição não muda mais
    final, r = rotulos[-1], len(rotulos)
    while r > 1 and rotulos[r - 2] == final:
        r -= 1
    return r


def media_pond(itens, chave):
    peso = sum(i["participantes"] for i in itens)
    return sum(i[chave] * i["participantes"] for i in itens) / peso if peso else None


def metricas(chave):
    alpha, frac, seed, drop = chave
    b = BRUTOS[str(execucoes[chave])]
    hist = HIST[(alpha, seed)]
    p_cli = hist / hist.sum(1, keepdims=True)
    visto, linhas = set(), []
    for r in range(ROUNDS):
        dono = np.array(b["rotulos_por_rodada"][r])
        visto |= set(b["sorteados_por_rodada"][r])
        _, inv = np.unique(dono, return_inverse=True)
        tam = np.bincount(inv)
        hc = np.zeros((len(tam), 10))
        np.add.at(hc, inv, hist)
        p_clu = hc / hc.sum(1, keepdims=True)
        itens = []
        for i in b["acuracia_por_rodada"][r]:
            p = p_clu[inv[i["cluster"]]]
            itens.append({**i, "pond_cluster": float(np.dot(p, i["recall_cluster"])), "pond_global": float(np.dot(p, i["recall_global"]))})
        linhas.append({"rodada": r + 1, "clusters": len(tam), "maior": int(tam.max()), "singletons": int((tam == 1).sum()),
                       "tvd": float(0.5 * np.abs(p_cli - p_clu[inv]).sum(1).mean()), "cobertura": len(visto) / N,
                       "participantes": len(b["sorteados_por_rodada"][r]), "disponiveis": 1 - len(b["indisponiveis_por_rodada"][r]) / N,
                       "avaliados": len(itens), "acc_cluster": media_pond(itens, "acc_cluster"), "acc_global": media_pond(itens, "acc_global"),
                       "acc_cluster_pond": media_pond(itens, "pond_cluster"), "acc_global_pond": media_pond(itens, "pond_global")})
    df = pd.DataFrame(linhas)
    fim = df.tail(JANELA_FINAL)
    acima = df.index[df["acc_cluster"].fillna(-1) >= LIMIAR_ACC]
    u = df.iloc[-1]
    final = {"alpha": alpha, "m": frac, "drop": drop, "seed": seed, "pasta": Path(execucoes[chave]).name,
             **{f"{c}_final": float(fim[c].mean()) for c in ("acc_cluster", "acc_global", "acc_cluster_pond", "acc_global_pond")},
             "rodada_limiar": int(df.loc[acima[0], "rodada"]) if len(acima) else None,
             "clusters": int(u["clusters"]), "maior": int(u["maior"]), "singletons": int(u["singletons"]), "tvd": float(u["tvd"]),
             "estavel": rodada_estavel(b["rotulos_por_rodada"]), "cobertura": float(u["cobertura"]),
             "participantes": float(df["participantes"].mean()), "disponiveis": float(df["disponiveis"].mean())}
    final["ganho"] = final["acc_cluster_final"] - final["acc_global_final"]
    final["ganho_pond"] = final["acc_cluster_pond_final"] - final["acc_global_pond_final"]
    for col, v in (("drop", drop), ("m", frac), ("seed", seed), ("alpha", alpha)):
        df.insert(0, col, v)
    return df, final


tabelas, finais = [], []
for chave in execucoes:
    df, f = metricas(chave)
    tabelas.append(df)
    finais.append(f)
por_rodada = pd.concat(tabelas, ignore_index=True)
finais = pd.DataFrame(finais)

colunas_resumo = ["acc_cluster_final", "acc_global_final", "ganho", "acc_cluster_pond_final", "acc_global_pond_final", "ganho_pond",
                  "rodada_limiar", "clusters", "maior", "singletons", "tvd", "estavel", "cobertura", "participantes", "disponiveis"]
resumo = finais.groupby(["alpha", "m", "drop"])[colunas_resumo].mean().reset_index()
resumo.insert(3, "execucoes", finais.groupby(["alpha", "m", "drop"]).size().values)
resumo.insert(4, "seeds_sem_limiar", finais.groupby(["alpha", "m", "drop"])["rodada_limiar"].apply(lambda s: int(s.isna().sum())).values)
resumo.insert(2, "cenário", resumo.pop("drop").map(nome))
resumo["m"] = resumo["m"].map(lambda f: f"{f:.0%}")
display(resumo.round(3))

### Dropout × sem dropout, seed a seed

Para cada seed, a diferença `dropout − sem dropout` com o **teste de Wilcoxon** dos pares. As seeds são pareadas pela partição dos dados e pelo modelo inicial, **não pelo sorteio** (que diverge assim que alguém cai). Com 10 seeds o teste tem pouco poder, e os p-valores são orientativos (sem correção para comparações múltiplas). `rodada_limiar` só entra nas seeds em que as duas execuções chegaram ao limiar.

In [ ]:
METRICAS_COMP = [("acc_cluster_final", "acc cluster"), ("acc_global_final", "acc global"), ("ganho", "ganho"), ("ganho_pond", "ganho ponderado"),
                 ("rodada_limiar", "rodada limiar"), ("estavel", "rodada estável"), ("clusters", "nº de clusters")]
linhas = []
for (alpha, frac), g in finais.groupby(["alpha", "m"]):
    base = g[g["drop"] == 0].set_index("seed")
    for d in sorted(set(g["drop"]) - {0.0}):
        com = g[g["drop"] == d].set_index("seed")
        linha = {"α": alpha, "m": f"{frac:.0%}", "cenário": nome(d)}
        for col, rotulo in METRICAS_COMP:
            dif = (com[col] - base.reindex(com.index)[col]).astype(float).dropna()
            linha[f"Δ {rotulo}"] = round(dif.mean(), 4) if len(dif) else None
            linha[f"p ({rotulo})"] = None if len(dif) == 0 else (1.0 if (dif == 0).all() else round(wilcoxon(dif).pvalue, 3))
        linha["seeds"] = len(com.index.intersection(base.index))
        linhas.append(linha)
comparacao = pd.DataFrame(linhas)
display(comparacao)

## 3. Gráficos por rodada

Uma figura por `α`, um painel por métrica (linhas) e por `m` (colunas). A linha é a **média entre as seeds** e a faixa vai do mínimo ao máximo; clique na legenda para ocultar um cenário. As acurácias são as do teste balanceado (painéis "acc") e ponderadas pela distribuição de rótulos do cluster ("acc ponderada").

In [ ]:
CORES_D = ["#2a78d6", "#eb6834", "#1baf7a", "#8a5cd0", "#eda100"]
TRACOS_D = ["solid", "dash", "dot", "dashdot", "longdash"]
cor = lambda d: CORES_D[DROPS.index(d) % len(CORES_D)]
traco = lambda d: TRACOS_D[DROPS.index(d) % len(TRACOS_D)]
paineis = [("clusters", "Nº de clusters"), ("maior", "Maior cluster"), ("acc_cluster", "Acc do modelo do cluster"),
           ("acc_global", "Acc do modelo global"), ("acc_cluster_pond", "Acc ponderada do cluster"), ("participantes", "Participantes na rodada")]

for alpha in ALPHAS:
    d_a = por_rodada[por_rodada["alpha"] == alpha]
    titulos = [f"{t}, m = {f:.0%}" for _, t in paineis for f in FRACOES]
    fig = make_subplots(rows=len(paineis), cols=len(FRACOES), subplot_titles=titulos, horizontal_spacing=0.08, vertical_spacing=0.05)
    for i, (col, _) in enumerate(paineis, start=1):
        for j, frac in enumerate(FRACOES, start=1):
            for d in DROPS:
                g = d_a[(d_a["m"] == frac) & (d_a["drop"] == d)].groupby("rodada")[col]
                media = g.mean()
                leg = nome(d)
                fig.add_trace(go.Scatter(x=list(media.index) + list(media.index[::-1]), y=list(g.max()) + list(g.min()[::-1]),
                                         fill="toself", fillcolor=cor(d), opacity=0.12, line=dict(width=0), hoverinfo="skip",
                                         showlegend=False, legendgroup=leg), row=i, col=j)
                fig.add_trace(go.Scatter(x=media.index, y=media.values, mode="lines", name=leg, legendgroup=leg,
                                         showlegend=(i == 1 and j == 1), line=dict(color=cor(d), dash=traco(d), width=2),
                                         hovertemplate="rodada %{x}: %{y:.3g}<extra>" + leg + "</extra>"), row=i, col=j)
            fig.update_xaxes(title_text="rodada" if i == len(paineis) else None, row=i, col=j)
    fig.update_layout(title=f"α = {alpha}, τ = {TAU}, {len(SEEDS)} seeds (média e faixa mín–máx)", height=220 * len(paineis) + 120,
                      hovermode="x unified", margin=dict(t=90, b=40, l=50, r=20), legend=dict(orientation="h", y=-0.04), template="simple_white")
    fig.update_xaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
    fig.update_yaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
    fig.show()

## 4. Resultado final por seed

Cada ponto é uma seed; a caixa resume as seeds de cada cenário. `ganho` negativo não é defeito: no teste balanceado, um modelo especializado em poucas classes perde para o global por construção (plano, seção 7); o `ganho ponderado` corrige isso pela distribuição de rótulos do cluster.

In [ ]:
metricas_final = [("acc_cluster_final", "Acc do cluster (final)"), ("acc_global_final", "Acc global (final)"), ("ganho", "Ganho"),
                  ("ganho_pond", "Ganho ponderado"), ("clusters", "Nº de clusters finais")]
for alpha in ALPHAS:
    f_a = finais[finais["alpha"] == alpha]
    titulos = [f"{t}, m = {f:.0%}" for f in FRACOES for _, t in metricas_final]
    fig = make_subplots(rows=len(FRACOES), cols=len(metricas_final), subplot_titles=titulos, horizontal_spacing=0.05, vertical_spacing=0.16)
    for i, frac in enumerate(FRACOES, start=1):
        for j, (col, _) in enumerate(metricas_final, start=1):
            for d in DROPS:
                v = f_a[(f_a["m"] == frac) & (f_a["drop"] == d)]
                fig.add_trace(go.Box(y=v[col], x=[nome(d)] * len(v), name=nome(d), marker=dict(color=cor(d)), boxpoints="all", jitter=0.4,
                                     pointpos=0, showlegend=(i == 1 and j == 1), legendgroup=nome(d), text=[f"seed {s}" for s in v["seed"]],
                                     hovertemplate="%{text}: %{y:.4g}<extra>" + nome(d) + "</extra>"), row=i, col=j)
    fig.update_layout(title=f"α = {alpha}, τ = {TAU}: uma seed por ponto", height=340 * len(FRACOES) + 100,
                      margin=dict(t=90, b=40, l=50, r=20), legend=dict(orientation="h", y=-0.08), template="simple_white")
    fig.update_yaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
    fig.show()

## 5. Cluster × classe da partição final

Para cada `α` e `m`, um painel por cenário com os `TOP_CLUSTERS` maiores clusters da partição final (colunas, do maior para o menor) e a proporção de cada classe (linhas) nos dados do cluster; o menu troca a seed. Um cluster especializado tem uma coluna concentrada em poucas classes; o rótulo de cada coluna traz o tamanho do cluster.

In [ ]:
TOP_CLUSTERS = 20
for alpha in ALPHAS:
    for frac in FRACOES:
        fig = make_subplots(rows=1, cols=len(DROPS), subplot_titles=[nome(d) for d in DROPS], horizontal_spacing=0.05)
        for s in SEEDS:
            for j, d in enumerate(DROPS, start=1):
                dono = np.array(BRUTOS[str(execucoes[(alpha, frac, s, d)])]["rotulos_por_rodada"][-1])
                ids, tam = np.unique(dono, return_counts=True)
                ordem = ids[np.argsort(-tam, kind="stable")][:TOP_CLUSTERS]
                hist = HIST[(alpha, s)]
                Z = np.array([hist[dono == c].sum(0) / hist[dono == c].sum() for c in ordem]).T
                rot = [f"{c} ({(dono == c).sum()})" for c in ordem]
                fig.add_trace(go.Heatmap(z=Z, x=list(range(1, len(ordem) + 1)), y=list(range(10)), colorscale="Blues", zmin=0, zmax=1,
                                         showscale=(j == len(DROPS)), colorbar=dict(title="proporção"), visible=(s == SEEDS[0]),
                                         customdata=np.array([rot] * 10), hovertemplate="cluster %{customdata}: classe %{y}, %{z:.2f}<extra></extra>"),
                              row=1, col=j)
        for j in range(1, len(DROPS) + 1):
            fig.update_xaxes(title_text=f"{TOP_CLUSTERS} maiores clusters", row=1, col=j)
            fig.update_yaxes(title_text="classe" if j == 1 else None, tickvals=list(range(10)), autorange="reversed", row=1, col=j)
        botoes = [dict(label=f"seed {s}", method="update", args=[{"visible": [t == s for t in SEEDS for _ in DROPS]}]) for s in SEEDS]
        fig.update_layout(title=f"Cluster × classe, α = {alpha}, m = {frac:.0%} (rodada {ROUNDS})", height=480, margin=dict(t=110, b=40, l=60, r=20),
                          updatemenus=[dict(buttons=botoes, direction="down", x=0.0, xanchor="left", y=1.25, yanchor="top")], template="simple_white")
        fig.show()

## 6. Leitura

- **Clusterização igual a `--train 0`.** O treino não muda o sorteio nem a partição (plano, seção 3; conferido no smoke test). O nº de clusters e o efeito do dropout neles repetem o Monte Carlo do `varredura-tau-dirichlet.ipynb`; o que esta grade acrescenta é a acurácia.
- **Com `τ = 0.84683` os clusters são pequenos** (~110 clusters, ~82 singletons em média no Monte Carlo): a acurácia por cluster tem viés de composição, porque só clusters sorteados são avaliados e muitos têm um cliente só. Leia junto com o nº e o tamanho dos clusters.
- **Ganho negativo no teste balanceado é esperado** para modelos especializados; compare com o ganho ponderado.
- **Efeito possivelmente nulo.** O dropout só atrasa a observação dos clientes (guia, 4.11); o efeito na acurácia pode aparecer só nas primeiras rodadas, ou não aparecer.
- **FedAvg sem ponderação** (D-045): clientes de 51 a 709 amostras pesam igual no `gmodel` e no modelo do cluster.

## 7. Resultados em JSON

Grava `logs/resultados-dropout-treino.json` (estrito, via arquivo temporário; no modo `DADOS_JSON` não grava nada):

- `parametros`; `histogramas`: por `(α, seed)`, o histograma de rótulos de cada cliente (`N` × 10).
- `execucoes`: uma por execução, com as métricas finais e os dados brutos: `rotulos_por_rodada` (o cluster de cada cliente em cada rodada), `sorteados_por_rodada`, `indisponiveis_por_rodada`, `acuracia_por_rodada` (por rodada, cada cluster avaliado: id, participantes, acurácia do cluster e do global, recall por classe dos dois) e `cossenos_entre_clusters` (última rodada, do `cos_rec.log`), mais a duração e a conferência da cobertura contra o `psi.pkl` (`None` se o `psi.pkl` já tinha sido apagado).
- `por_rodada`, `resumo` e `comparacao`: as tabelas das seções 2 e 3.

In [ ]:
def registros(df):
    # precisão completa e tipos do Python; NaN vira None (JSON estrito)
    return [{k: (None if isinstance(v, float) and np.isnan(v) else (v.item() if hasattr(v, "item") else v)) for k, v in r.items()}
            for r in df.to_dict("records")]


if DADOS_JSON:
    print(f"dados lidos de {RESULTADOS_JSON.name}: nada a gravar")
else:
    por_pasta = {Path(p).name: p for p in execucoes.values()}
    resultado = {
        "gerado_em": datetime.now().isoformat(timespec="seconds"),
        "parametros": parametros_atuais(),
        "histogramas": [{"alpha": a, "seed": s, "histograma": HIST[(a, s)].tolist()} for a in ALPHAS for s in SEEDS],
        "execucoes": [{**l, **BRUTOS[str(por_pasta[l["pasta"]])], "duracao_s": DURACOES.get(str(por_pasta[l["pasta"]])),
                       "cobertura_igual_psi": COBERTURA_PSI.get(str(por_pasta[l["pasta"]]))} for l in registros(finais)],
        "por_rodada": registros(por_rodada),
        "resumo": registros(resumo),
        "comparacao": registros(comparacao),
    }
    tmp = RESULTADOS_JSON.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(resultado, ensure_ascii=False, allow_nan=False))
    tmp.replace(RESULTADOS_JSON)
    print(f"{RESULTADOS_JSON} ({RESULTADOS_JSON.stat().st_size / 1e6:.1f} MB): {len(resultado['execucoes'])} execuções, {len(resultado['por_rodada'])} linhas por rodada")